# Day 28 — Does the tradability filter drop the thinnest names?

**Acceptance:** the tradability filter correctly excludes the thinnest names in the universe.

A stock is *tradable* at session $t$ when its average daily euro volume over the last 21
sessions clears €5M:

$$\text{adv\_eur\_21}(t) = \frac{1}{21}\sum_{k=0}^{20} \text{volume}_{t-k}\cdot\text{close\_eur}_{t-k} \;\ge\; €5\,000\,000$$

"Correctly" means three things, each checked below on the real store:

1. `adv_eur_21` is what it says (a hand calculation matches).
2. On every date, the excluded names are exactly those under the threshold, so every
   excluded name is thinner than every included one: the filter cuts the bottom of the
   cross-section and nothing else.
3. The names it does exclude are the ones that really were thin, and the threshold is
   not dead weight on a universe of 100 large caps.

In [1]:
import datetime as dt

from finmgr.data import store
from finmgr.features.liquidity import MIN_ADV_EUR, compute_liquidity, read_liquidity, thinnest

AS_OF = "2026-10-15"  # pinned so the numbers below are reproducible
t = dt.date.fromisoformat(AS_OF)

panel = read_liquidity(as_of=AS_OF)
print(
    f"{len(panel):,} rows, {panel['ticker'].nunique()} tickers, "
    f"{panel['date'].min()} to {panel['date'].max()}"
)

416,976 rows, 100 tickers, 2010-01-04 to 2026-10-15


## Step 1 — the hand calculation

The thinnest name today, straight from the raw bars: shares times the euro close, last 21 sessions.

In [2]:
today = panel[panel["date"] == t]
name = today.sort_values("adv_eur_21")["ticker"].iloc[0]
bars = store.read_bars(name, as_of=AS_OF).sort_values("date").tail(21)
hand = float((bars["volume"] * bars["close_eur"]).mean())
mod = float(today.set_index("ticker").loc[name, "adv_eur_21"])
print(f"{name}: last 21 bars {bars['date'].iloc[0]} to {bars['date'].iloc[-1]}")
print(f"adv_eur_21 by hand €{hand:,.0f}   module €{mod:,.0f}")
assert abs(mod / hand - 1) < 1e-12
print("MATCH")

GRF.MC: last 21 bars 2026-09-17 to 2026-10-15
adv_eur_21 by hand €12,123,027   module €12,123,027
MATCH


## Step 2 — today's thinnest names

The universe is 100 large caps, so on the pinned date nothing is under €5M: the thinnest
clears the bar by more than 2x. The flag is right to keep them all.

In [3]:
bottom = thinnest(panel, t, n=10)
print(bottom.assign(adv_eur_21=bottom["adv_eur_21"].map("€{:,.0f}".format)).to_string())
print(f"median of the 100: €{today['adv_eur_21'].median():,.0f}")
assert today["adv_eur_21"].notna().all()
assert today["tradable"].all() == (today["adv_eur_21"] >= MIN_ADV_EUR).all()

    ticker   adv_eur_21  tradable
0   GRF.MC  €12,123,027      True
1  0823.HK  €26,919,702      True
2  PHIA.AS  €33,382,929      True
3  AENA.MC  €33,519,409      True
4   ACS.MC  €38,246,011      True
5  HEIA.AS  €38,441,398      True
6   AMS.MC  €39,375,418      True
7   KPN.AS  €45,757,353      True
8   TEF.MC  €47,568,652      True
9   VNA.DE  €62,082,658      True
median of the 100: €163,901,742


## Step 3 — every date: the excluded are exactly the bottom of the cross-section

Over the whole history, on each date: every name with `tradable == False` and a known
average is under €5M; every included name is over it; and so the thinnest included name
is always more liquid than the thickest excluded one. A name without enough history for
an average (a new listing) is never tradable.

In [4]:
known = panel.dropna(subset=["adv_eur_21"])
assert (known["tradable"] == (known["adv_eur_21"] >= MIN_ADV_EUR)).all()
assert not panel.loc[panel["adv_eur_21"].isna(), "tradable"].any()

by_day = known.groupby(["date", "tradable"])["adv_eur_21"].agg(["min", "max"]).unstack()
both = by_day.dropna()
assert (both[("max", False)] < both[("min", True)]).all()

# Rank within each day: an excluded name is always among the very thinnest.
known = known.assign(rank=known.groupby("date")["adv_eur_21"].rank(method="first"))
out = known[~known["tradable"]]
print(
    f"{len(both)} dates where the filter excluded at least one name; "
    f"on each, max excluded < min included"
)
print(
    f"rank of an excluded name within its day: at most {int(out['rank'].max())} "
    f"of ~{int(known.groupby('date').size().median())}"
)
assert out["rank"].max() <= 3

331 dates where the filter excluded at least one name; on each, max excluded < min included


rank of an excluded name within its day: at most 2 of ~99


## Step 4 — who it excluded, and whether they really were thin

The absolute threshold binds on a stock's own past, not on today's 100. The names and
periods it catches are the ones you would expect: Hermès in 2010-14, when its float was
small and LVMH still held a block of it; Vonovia in the year after its July 2013 IPO; and
Tesla a few months after its June 2010 IPO. Broadcom's single day in April 2010 sat just
under the line (€4.97M). Each was far below the median of the universe on the same days.

In [5]:
summary = (
    out.groupby("ticker")
    .agg(
        sessions=("date", "size"),
        first=("date", "min"),
        last=("date", "max"),
        lowest_adv=("adv_eur_21", "min"),
    )
    .sort_values("sessions", ascending=False)
)
median_that_day = known.groupby("date")["adv_eur_21"].median()
summary["median_of_universe_then"] = [
    median_that_day.loc[out.loc[out["ticker"] == tk, "date"]].median() for tk in summary.index
]
print(
    summary.to_string(
        formatters=dict.fromkeys(("lowest_adv", "median_of_universe_then"), "€{:,.0f}".format)
    )
)
assert {"RMS.PA", "VNA.DE"} <= set(summary.index)
assert (summary["lowest_adv"] < summary["median_of_universe_then"] / 10).all()

        sessions       first        last lowest_adv median_of_universe_then
ticker                                                                     
RMS.PA       304  2010-03-10  2014-09-02 €2,428,807            €175,901,184
VNA.DE       184  2013-08-09  2014-05-19   €340,591            €172,423,045
TSLA           4  2010-10-29  2010-11-03 €4,348,109            €168,141,586
AVGO           1  2010-04-13  2010-04-13 €4,969,044            €173,306,250


## Step 5 — a stricter bar still cuts from the bottom

Raising the threshold must exclude the thinnest k names and only them. On the pinned
date, at €25M, €50M and €100M, the excluded set equals the bottom k of the ranking.

In [6]:
ranked = today.sort_values("adv_eur_21")["ticker"].tolist()
for bar in (25e6, 50e6, 100e6):
    flagged = compute_liquidity(
        store.read_bars(as_of=AS_OF, start="2026-08-01").merge(
            panel[["ticker", "date", "adj_close"]], on=["ticker", "date"]
        ),
        min_adv_eur=bar,
    )
    day = flagged[flagged["date"] == t]
    excluded = set(day.loc[~day["tradable"], "ticker"])
    assert excluded == set(ranked[: len(excluded)]), bar
    names = ", ".join(ranked[: len(excluded)])
    print(f"€{bar / 1e6:>4.0f}M: excludes the bottom {len(excluded):>2}: {names}")
print("EXCLUDES THE THINNEST")

€  25M: excludes the bottom  1: GRF.MC


€  50M: excludes the bottom  9: GRF.MC, 0823.HK, PHIA.AS, AENA.MC, ACS.MC, HEIA.AS, AMS.MC, KPN.AS, TEF.MC


€ 100M: excludes the bottom 16: GRF.MC, 0823.HK, PHIA.AS, AENA.MC, ACS.MC, HEIA.AS, AMS.MC, KPN.AS, TEF.MC, VNA.DE, CAP.PA, RWE.DE, DSFIR.AS, AD.AS, ERIC-B.ST, BAS.DE
EXCLUDES THE THINNEST
